# Interactive Data Visualization with Matplotlib and Seaborn
## US Superstore Dataset

**Objectives:**
1. Load, clean, and preprocess the US Superstore dataset
2. Build interactive visualizations with Matplotlib (line chart + map)
3. Build communicative static charts with Seaborn (bar chart + scatter)
4. Comparative analysis of both libraries

---

## 1. Data Preparation

In [ ]:
# ── Core libraries ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider, SelectMultiple
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ── Optional mapping library ───────────────────────────────────────────────────
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False
    print('geopandas not available - US state map will use a built-in Matplotlib alternative.')

# ── Global style ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print('Libraries loaded successfully.')

### 1.1 Loading the Dataset

In [ ]:
URL = ('https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/'
       'Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/'
       'US%20Superstore%20data.xls')

try:
    df = pd.read_excel(URL)
    print('Loaded from remote URL.')
except Exception:
    df = pd.read_csv('superstore_dataset.csv', encoding='latin-1')
    print('Loaded from local CSV.')

print(f'Shape: {df.shape}')
df.head(3)

### 1.2 Basic Exploration

In [ ]:
print('Columns:', df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
df.describe()

### 1.3 Cleaning and Preprocessing

Steps applied:
- **Duplicates**: removed unconditionally (each order line should be unique).
- **Postal Code nulls**: filled with `0` — geographic analysis uses State, not postal codes.
- **Date columns**: converted to `datetime` to unlock `.dt` accessors.
- **Feature engineering**: `Profit Margin`, `Order Year`, `Order Month`, `Order Month-Year`.

In [ ]:
# Remove duplicates
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicates after: ', df.duplicated().sum())

# Fill missing Postal Code
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0).astype(int)

# Date conversion
for col in ['Order Date', 'Ship Date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

# Feature engineering
df['Profit Margin']    = (df['Profit'] / df['Sales']) * 100
df['Order Year']       = df['Order Date'].dt.year
df['Order Month']      = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print('\nPreprocessing complete. Sample:')
df[['Order Date', 'Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head()

---
## 2. Matplotlib Visualizations

### 2.1 Interactive Line Chart - Sales Trend Over the Years

Two widgets control the chart:
- **Category dropdown**: filters by product category (or shows all).
- **Granularity toggle**: switches between monthly and yearly aggregation.

**Reading the chart:**
- Look for a consistent upward slope as a signal of year-over-year growth.
- Recurring spikes in the same months each year reveal seasonality (often Q4 holiday effect).
- Divergence across categories highlights which segments are driving overall growth.

In [ ]:
# Prepare aggregations used by the interactive chart
monthly_by_cat = (
    df.groupby(['Order Month-Year', 'Category'])['Sales']
    .sum()
    .reset_index()
)
monthly_by_cat['Date'] = monthly_by_cat['Order Month-Year'].dt.to_timestamp()

monthly_all = (
    df.groupby('Order Month-Year')['Sales']
    .sum()
    .reset_index()
)
monthly_all['Date'] = monthly_all['Order Month-Year'].dt.to_timestamp()

yearly_by_cat = (
    df.groupby(['Order Year', 'Category'])['Sales']
    .sum()
    .reset_index()
)
yearly_all = df.groupby('Order Year')['Sales'].sum().reset_index()

CATEGORY_COLORS = {
    'All':             '#2980b9',
    'Furniture':       '#e67e22',
    'Office Supplies': '#27ae60',
    'Technology':      '#8e44ad',
}

In [ ]:
def plot_sales_trend(category='All', granularity='Monthly'):
    fig, ax = plt.subplots(figsize=(14, 6))
    color = CATEGORY_COLORS.get(category, '#2980b9')

    if granularity == 'Monthly':
        if category == 'All':
            x, y = monthly_all['Date'], monthly_all['Sales']
        else:
            sub = monthly_by_cat[monthly_by_cat['Category'] == category]
            x, y = sub['Date'], sub['Sales']
        ax.set_xlabel('Month')
    else:  # Yearly
        if category == 'All':
            x, y = yearly_all['Order Year'], yearly_all['Sales']
        else:
            sub = yearly_by_cat[yearly_by_cat['Category'] == category]
            x, y = sub['Order Year'], sub['Sales']
        ax.set_xlabel('Year')

    ax.plot(x, y, marker='o', linewidth=2.2, markersize=5, color=color)
    ax.fill_between(x, y, alpha=0.12, color=color)

    # Annotate max point
    idx_max = np.argmax(y.values)
    ax.annotate(
        f'  Peak\n  ${y.values[idx_max]:,.0f}',
        xy=(list(x)[idx_max], y.values[idx_max]),
        fontsize=9, color=color, fontweight='bold'
    )

    cat_label = category if category != 'All' else 'All Categories'
    ax.set_title(f'Sales Trend - {cat_label} ({granularity})', pad=14)
    ax.set_ylabel('Total Sales (USD)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
    ax.tick_params(axis='x', rotation=40)
    ax.grid(True, alpha=0.35)
    fig.tight_layout()
    plt.show()

    print(f'Total sales ({cat_label}): ${y.sum():,.0f}')
    print(f'Average per period:       ${y.mean():,.0f}')
    print(f'Growth (first to last):   {(y.values[-1]/y.values[0]-1)*100:+.1f}%')


category_widget    = Dropdown(
    options=['All'] + sorted(df['Category'].unique().tolist()),
    value='All', description='Category:'
)
granularity_widget = Dropdown(
    options=['Monthly', 'Yearly'],
    value='Monthly', description='Granularity:'
)

interact(plot_sales_trend, category=category_widget, granularity=granularity_widget);

### 2.2 Interactive Map - Sales Distribution by US State

The map encodes total sales as both colour intensity and bubble size. Two variants are provided:
- **Variant A (geopandas)**: choropleth with full state polygons if `geopandas` is installed.
- **Variant B (built-in)**: bubble map using approximate state centroids — no extra dependencies required.

The interactive slider controls the colour scale upper bound, making it possible to highlight mid-tier states that would otherwise be washed out by California and New York.

In [ ]:
# State-level aggregation
state_sales = (
    df.groupby('State')[['Sales', 'Profit']]
    .sum()
    .assign(**{'Profit Margin': lambda x: x['Profit'] / x['Sales'] * 100})
    .reset_index()
)

# Approximate centroids for all 48 contiguous states + DC
STATE_COORDS = {
    'Alabama': (32.8, -86.8), 'Arizona': (34.3, -111.1), 'Arkansas': (34.8, -92.2),
    'California': (36.8, -119.4), 'Colorado': (39.0, -105.5), 'Connecticut': (41.6, -72.7),
    'Delaware': (38.9, -75.5), 'District of Columbia': (38.9, -77.0),
    'Florida': (27.8, -81.7), 'Georgia': (32.2, -83.4), 'Idaho': (44.2, -114.5),
    'Illinois': (40.3, -89.0), 'Indiana': (39.8, -86.1), 'Iowa': (42.0, -93.2),
    'Kansas': (38.5, -98.4), 'Kentucky': (37.5, -85.3), 'Louisiana': (31.1, -91.8),
    'Maine': (45.4, -69.0), 'Maryland': (39.1, -76.8), 'Massachusetts': (42.3, -71.8),
    'Michigan': (44.3, -85.4), 'Minnesota': (46.4, -93.1), 'Mississippi': (32.7, -89.7),
    'Missouri': (38.5, -92.3), 'Montana': (47.0, -110.5), 'Nebraska': (41.5, -99.9),
    'Nevada': (38.5, -116.4), 'New Hampshire': (43.7, -71.6), 'New Jersey': (40.1, -74.5),
    'New Mexico': (34.5, -105.9), 'New York': (42.9, -75.6), 'North Carolina': (35.6, -79.4),
    'North Dakota': (47.5, -100.5), 'Ohio': (40.4, -82.8), 'Oklahoma': (35.6, -97.5),
    'Oregon': (44.0, -120.5), 'Pennsylvania': (40.9, -77.8), 'Rhode Island': (41.7, -71.5),
    'South Carolina': (33.8, -80.9), 'South Dakota': (44.4, -100.2),
    'Tennessee': (35.9, -86.4), 'Texas': (31.5, -99.3), 'Utah': (39.3, -111.1),
    'Vermont': (44.0, -72.7), 'Virginia': (37.8, -78.2), 'Washington': (47.4, -120.4),
    'West Virginia': (38.6, -80.6), 'Wisconsin': (44.3, -89.8), 'Wyoming': (43.0, -107.6),
}

state_sales['Lat'] = state_sales['State'].map(lambda s: STATE_COORDS.get(s, (np.nan, np.nan))[0])
state_sales['Lon'] = state_sales['State'].map(lambda s: STATE_COORDS.get(s, (np.nan, np.nan))[1])
state_sales = state_sales.dropna(subset=['Lat', 'Lon'])

print(f'States mapped: {len(state_sales)}')
state_sales.sort_values('Sales', ascending=False).head(5)

In [ ]:
def plot_sales_map(metric='Sales', scale_cap_pct=100):
    """
    Bubble map of US state performance.
    scale_cap_pct: clip the colour scale at this percentile to reveal mid-tier states.
    """
    fig, ax = plt.subplots(figsize=(16, 9))
    ax.set_facecolor('#d4eaf7')
    fig.patch.set_facecolor('#f0f4f8')

    # Draw a simple US bounding box as context
    us_box = plt.Polygon(
        [(-125, 24), (-66, 24), (-66, 50), (-125, 50)],
        fill=False, edgecolor='#aaaaaa', linewidth=1, linestyle='--'
    )
    ax.add_patch(us_box)

    values = state_sales[metric].values
    vmax   = np.percentile(values, scale_cap_pct)
    vmin   = values.min()

    norm  = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap  = plt.cm.YlOrRd

    # Bubble size proportional to metric value
    sizes = ((values - vmin) / (vmax - vmin + 1e-9)) * 1800 + 80

    sc = ax.scatter(
        state_sales['Lon'], state_sales['Lat'],
        c=values, cmap=cmap, norm=norm,
        s=sizes, alpha=0.82, edgecolors='white', linewidths=0.6, zorder=3
    )

    # Label the top-5 states
    top5 = state_sales.nlargest(5, metric)
    for _, row in top5.iterrows():
        unit = '$' if metric in ['Sales', 'Profit'] else ''
        val  = f"{unit}{row[metric]:,.0f}" if metric != 'Profit Margin' else f"{row[metric]:.1f}%"
        ax.annotate(
            f"{row['State'].split()[0]}\n{val}",
            xy=(row['Lon'], row['Lat']),
            xytext=(6, 6), textcoords='offset points',
            fontsize=8, fontweight='bold', color='#1a1a2e',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.7, ec='none')
        )

    plt.colorbar(sc, ax=ax, fraction=0.025, pad=0.02,
                 label=f'{metric} (USD)' if metric != 'Profit Margin' else 'Profit Margin (%)')

    ax.set_xlim(-128, -63)
    ax.set_ylim(22, 52)
    ax.set_title(f'US Superstore - {metric} by State\n(top 5 states labelled; colour scale capped at {scale_cap_pct}th percentile)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

    print(f'Top 5 states by {metric}:')
    for _, row in top5.iterrows():
        print(f"  {row['State']:<25} {metric}: {row[metric]:>10,.1f}")


metric_widget = Dropdown(
    options=['Sales', 'Profit', 'Profit Margin'],
    value='Sales', description='Metric:'
)
cap_slider = IntSlider(
    min=50, max=100, value=100, step=5,
    description='Scale cap %:',
    style={'description_width': 'initial'}
)

interact(plot_sales_map, metric=metric_widget, scale_cap_pct=cap_slider);

---
## 3. Seaborn Visualizations

### 3.1 Top 10 Products by Sales

Seaborn's `barplot` is used here for its built-in confidence-interval rendering and clean palette management. The chart is styled for an executive audience: horizontal orientation maximises label readability, and direct annotations remove the need to consult the axis scale.

In [ ]:
top10_products = (
    df.groupby('Product Name')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top10_products.columns = ['Product Name', 'Total Sales']

# Truncate long product names for readability
top10_products['Label'] = top10_products['Product Name'].str[:55]

fig, ax = plt.subplots(figsize=(14, 8))

sns.barplot(
    data=top10_products,
    x='Total Sales', y='Label',
    palette='Blues_r', orient='h', ax=ax
)

# Direct value annotations
for i, row in top10_products.iterrows():
    ax.text(
        row['Total Sales'] + top10_products['Total Sales'].max() * 0.008,
        i, f'${row["Total Sales"]:,.0f}',
        va='center', fontsize=9.5, fontweight='bold', color='#2c3e50'
    )

ax.set_title('Top 10 Products by Total Sales\nExecutive Overview', pad=18)
ax.set_xlabel('Total Sales (USD)')
ax.set_ylabel('Product')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.grid(axis='x', alpha=0.35)
fig.tight_layout()
plt.show()

print('Top 10 products represent '
      f'{top10_products["Total Sales"].sum() / df["Sales"].sum() * 100:.1f}% of total sales.')

### 3.2 Scatter Plot - Profit vs Discount

The scatter plot examines whether higher discounts systematically erode profitability. Key design choices:
- `hue=Category` surfaces differential sensitivity across product lines.
- `regplot` overlays an OLS trend line per category to quantify the slope.
- The break-even line (profit = 0) acts as a visual reference.

**What to look for:**
- The discount level at which the overall trend line crosses zero profit.
- Categories with steeper negative slopes are more sensitive to discounting.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

category_palette = {
    'Furniture':       '#e67e22',
    'Office Supplies': '#27ae60',
    'Technology':      '#8e44ad',
}

# Main scatter
sns.scatterplot(
    data=df, x='Discount', y='Profit',
    hue='Category', palette=category_palette,
    alpha=0.45, s=40, ax=ax, zorder=2
)

# Per-category regression lines
for category, color in category_palette.items():
    cat_df = df[df['Category'] == category]
    sns.regplot(
        data=cat_df, x='Discount', y='Profit',
        scatter=False, color=color,
        line_kws={'linewidth': 2.2, 'linestyle': '--'},
        ax=ax
    )

# Overall regression
sns.regplot(
    data=df, x='Discount', y='Profit',
    scatter=False, color='#c0392b',
    line_kws={'linewidth': 2.5, 'linestyle': '-', 'label': 'Overall trend'},
    ax=ax
)

# Break-even reference
ax.axhline(y=0, color='black', linewidth=1, alpha=0.5, linestyle='-')
ax.text(0.52, 20, 'Break-even', fontsize=9, alpha=0.6)

ax.set_title('Profit vs Discount Rate by Product Category', pad=16)
ax.set_xlabel('Discount Rate')
ax.set_ylabel('Profit (USD)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.grid(True, alpha=0.3)

# Rebuild legend to include overall trend line
handles, labels = ax.get_legend_handles_labels()
overall_patch = mpatches.Patch(color='#c0392b', label='Overall trend')
ax.legend(handles=handles + [overall_patch],
          labels=labels + ['Overall trend'],
          title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')

fig.tight_layout()
plt.show()

# Quantified insights
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]
print('Loss rate by discount threshold:')
for t in thresholds:
    bucket = df[df['Discount'] >= t]
    if len(bucket) > 0:
        loss_rate = (bucket['Profit'] < 0).mean() * 100
        avg_profit = bucket['Profit'].mean()
        print(f'  Discount >= {t*100:.0f}%: '
              f'{len(bucket):>5} transactions, '
              f'loss rate {loss_rate:>5.1f}%, '
              f'avg profit ${avg_profit:>8.2f}')

---
## 4. Comparative Analysis: Matplotlib vs Seaborn

### Summary Table

| Dimension | Matplotlib | Seaborn |
|---|---|---|
| **Control level** | Very high — every element is manually configurable | Medium — sensible defaults limit low-level overrides |
| **Interactivity** | Native integration with `ipywidgets` | Compatible, but stateful figure objects can cause display artefacts |
| **Statistical overlays** | Requires manual `numpy` implementation | Built-in `regplot`, `boxplot`, `violinplot`, confidence bands |
| **Default aesthetics** | Functional but minimal | Publication-ready out of the box |
| **Mapping** | Requires manual coordinate management | Not designed for geographic plots |
| **Learning curve** | Steeper; requires explicit `fig`/`ax` management | Shallower for standard chart types |
| **Best use case** | Interactive dashboards, custom maps, exploratory widget-driven analysis | Stakeholder presentations, statistical summaries, executive reports |

### Key Observations from This Notebook

**Matplotlib** was indispensable for the two interactive sections:
- The sales trend chart required a function that rebuilt the figure on each widget update. Matplotlib's imperative `fig, ax = plt.subplots()` pattern integrates cleanly with `interact()`.
- The bubble map required precise control over colour normalisation, bubble sizing, annotation positioning, and background colour — all things Seaborn's high-level API does not expose.

**Seaborn** was the better choice for the two communicative charts:
- The bar chart used `palette='Blues_r'` to encode rank with minimal code. Achieving the same gradient in raw Matplotlib requires a manual `plt.cm` call and a loop.
- The scatter plot used `hue` and `regplot` in a few lines to produce a multi-layer statistical chart. The Matplotlib equivalent would require iterating over categories, computing regression coefficients manually, and drawing each line separately.

**Recommendation:**
- Use Matplotlib when the chart needs to respond to user input or when fine-grained layout control is required.
- Use Seaborn when the goal is a polished, statistically informative chart for a report or presentation.

In [ ]:
import time

yearly_data = df.groupby('Order Year')['Sales'].sum().reset_index()

# Time Matplotlib
start = time.perf_counter()
for _ in range(20):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(yearly_data['Order Year'], yearly_data['Sales'])
    plt.close(fig)
t_mpl = (time.perf_counter() - start) / 20

# Time Seaborn
start = time.perf_counter()
for _ in range(20):
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.lineplot(data=yearly_data, x='Order Year', y='Sales', ax=ax)
    plt.close(fig)
t_sns = (time.perf_counter() - start) / 20

print(f'Average render time over 20 iterations:')
print(f'  Matplotlib: {t_mpl*1000:.1f} ms')
print(f'  Seaborn:    {t_sns*1000:.1f} ms')
print(f'  Overhead:   {t_sns/t_mpl:.1f}x (Seaborn adds theme/stat computation on top of Matplotlib)')

---
## 5. Key Analytical Findings

| Finding | Detail |
|---|---|
| **Sales growth** | Consistent year-over-year growth across all categories; strongest spike in Q4 each year |
| **Geographic concentration** | California and New York account for a disproportionate share of total revenue |
| **Discount risk** | Transactions with discounts above 20% show a materially higher loss rate |
| **Top products** | The top 10 products by sales are dominated by Technology (copiers, machines) |
| **Category sensitivity** | Furniture is the most discount-sensitive category — regression slope is steepest |

These findings translate directly into marketing priorities:
1. Protect Q4 margins — avoid deep discounts during peak demand periods.
2. Concentrate geographic investment in California and New York while monitoring margin.
3. Cap Furniture discounts at 20% and enforce approval workflows for exceptions.
4. Prioritise the top-10 products for inventory planning and promotional placement.